## 🔁 Comprehensive LangGraph Tutorial

This notebook provides an in-depth exploration of **LangGraph**, a library for building stateful, multi-agent workflows with Large Language Models (LLMs). It extends the original `Langgraph_basics.ipynb` by preserving its code, adding theoretical explanations, detailed comments, and introducing advanced functionalities. LangGraph is ideal for managing dynamic, cyclical, and interactive AI systems.

## What is LangGraph?
LangGraph is a framework for orchestrating complex, stateful agent workflows. It allows developers to define graphs where nodes represent agents or tools, and edges define the flow of execution. Key features include:
- **State Management**: Maintains a shared state across nodes.
- **Cyclical Flows**: Supports loops and conditional routing.
- **Memory Checkpoints**: Persists state for resumable workflows.
- **Human-in-the-Loop**: Integrates human feedback into the graph.
- **Custom Tools**: Extends functionality with user-defined tools.

## Objectives
- Demonstrate core LangGraph functionalities from the original notebook.
- Introduce advanced features like cycles, checkpoints, and human-in-the-loop.
- Provide theoretical context and detailed comments.
- Preserve and enhance the original code for calculator and customer support examples.

## Prerequisites
- Python 3.10+
- Install dependencies: `pip install langgraph langchain langchain-community langchain-ollama`
- Local Ollama server running with `mixtral` and `mistral` models.

## Structure
1. **Setup and Basic Graph (Calculator Example)**
2. **Simple Agent Graph (Customer Support Example)**
3. **Cyclical Workflows**
4. **Memory Checkpoints**
5. **Human-in-the-Loop Interactions**
6. **Custom State Management**
7. **Advanced Tool Integration**

Let's get started!

## 1. Setup and Basic Graph (Calculator Example)

**Theory**: LangGraph uses a `StateGraph` to define nodes (agents or tools) and edges (execution flow). The state is a shared data structure that persists across nodes. The original calculator example demonstrates a simple graph with an agent node (LLM) and a tool node (calculator), using conditional edges to determine the flow.

This section preserves the original calculator code with added comments and explanations.

In [ ]:
# Install required packages
!pip install langgraph langchain langchain-community langchain-ollama

In [ ]:
# Original Calculator Graph
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from typing import TypedDict, Annotated
import operator

# Define state schema
# Theory: The state is a TypedDict that tracks messages across nodes. Annotated[list, operator.add] ensures messages are appended.
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# Calculator tool
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression and returns the result as a string."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

# Setup LLM
# Theory: ChatOllama integrates with a local Ollama server for efficient, open-source LLM processing.
llm = ChatOllama(model="mixtral")

# Custom prompt for tool invocation
# Theory: The prompt instructs the LLM to flag math expressions for the calculator tool.
tool_prompt = PromptTemplate.from_template(
    "You are an assistant. If the user input is a math expression, say 'TOOL: {expression}'\nUser: {query}"
)

# Agent node
def agent(state: AgentState):
    """Processes user input using the LLM and determines if a tool is needed."""
    user_msg = state["messages"][-1]["content"]
    prompt = tool_prompt.format(query=user_msg, expression=user_msg)
    response = llm.invoke(prompt)
    return {"messages": [{"role": "assistant", "content": response.content}]}

# Tool node
def call_tool(state: AgentState):
    """Executes the calculator tool if the previous message indicates a math expression."""
    last_msg = state["messages"][-1]["content"]
    if last_msg.startswith("TOOL:"):
        expression = last_msg.replace("TOOL:", "").strip()
        result = calculate(expression)
        return {"messages": [{"role": "tool", "content": result}]}
    return {"messages": []}

# Create graph
# Theory: The graph defines nodes and edges. Conditional edges allow dynamic routing based on state.
graph = StateGraph(AgentState)
graph.add_node("agent", agent)
graph.add_node("tool", call_tool)
graph.add_edge("agent", "tool")
graph.add_conditional_edges("tool", lambda state: "agent" if state["messages"] else END)
graph.set_entry_point("agent")

# Compile and run
app = graph.compile()

# Test queries
queries = ["What is 5 * 3?", "What is LangGraph?"]
for query in queries:
    result = app.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"Query: {query}\nAnswer: {result['messages'][-1]['content']}\n")

## 2. Simple Agent Graph (Customer Support Example)

**Theory**: This example demonstrates a single-node graph for a customer support agent that maintains conversation history in the state. The original code uses a simple linear flow, ending after each agent response. We'll preserve this example and add comments for clarity.

In [ ]:
# Original Customer Support Graph
from langgraph.graph import StateGraph, END
from langchain_community.llms import Ollama
from typing import TypedDict, Annotated
import operator

# Define state schema
# Theory: The state tracks conversation history, enabling context-aware responses.
class SupportState(TypedDict):
    messages: Annotated[list, operator.add]

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent node
def agent(state: SupportState):
    """Generates a customer support response based on conversation history."""
    messages = state["messages"]
    prompt = f"You are a customer support agent. Use the conversation history to answer:\n{messages}"
    response = llm.invoke(prompt)
    return {"messages": [{"content": response, "role": "assistant"}]}

# Define graph
# Theory: A single-node graph with a direct edge to END creates a linear flow.
graph = StateGraph(SupportState)
graph.add_node("agent", agent)
graph.add_edge("agent", END)
graph.set_entry_point("agent")

# Compile graph
app = graph.compile()

# Run conversation
queries = [
    "What is the status of Alice's order?",
    "Can she get a refund?"
]
state = {"messages": []}
for query in queries:
    state = app.invoke({"messages": [{"content": query, "role": "user"}] + state["messages"]})
    print(f"Query: {query}\nAnswer: {state['messages'][-1]['content']}\n")

## 3. Cyclical Workflows

**Theory**: LangGraph supports cyclical workflows, allowing nodes to loop back based on conditions. This is useful for iterative tasks, such as refining answers or retrying failed operations. We'll create a new example where an agent refines its response until it meets a quality threshold.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from typing import TypedDict, Annotated
import operator

# Define state
class RefinementState(TypedDict):
    messages: Annotated[list, operator.add]
    iteration: int

# Setup LLM
llm = ChatOllama(model="mixtral")

# Agent node
def refine_agent(state: RefinementState):
    """Generates or refines a response to a user query."""
    user_msg = state["messages"][0]["content"]
    iteration = state["iteration"]
    prompt = PromptTemplate.from_template(
        "Refine this response for clarity and conciseness (Iteration {iteration}):\nQuery: {query}\nCurrent Response: {response}"
    )
    if iteration == 0:
        response = llm.invoke(f"Answer: {user_msg}")
    else:
        last_response = state["messages"][-1]["content"]
        response = llm.invoke(prompt.format(iteration=iteration, query=user_msg, response=last_response))
    return {"messages": [{"role": "assistant", "content": response.content}], "iteration": iteration + 1}

# Evaluator node
def evaluate_response(state: RefinementState):
    """Evaluates the response quality and decides whether to refine further."""
    response = state["messages"][-1]["content"]
    eval_prompt = PromptTemplate.from_template(
        "Is this response clear and concise? Answer 'Yes' or 'No':\n{response}"
    )
    evaluation = llm.invoke(eval_prompt.format(response=response)).content.strip()
    return {"messages": [{"role": "evaluator", "content": evaluation}]}

# Conditional routing
def route_refinement(state: RefinementState):
    """Routes to refine again if evaluation is 'No' or iteration < 3, else ends."""
    evaluation = state["messages"][-1]["content"]
    iteration = state["iteration"]
    if evaluation == "No" and iteration < 3:
        return "refine"
    return END

# Create graph
graph = StateGraph(RefinementState)
graph.add_node("refine", refine_agent)
graph.add_node("evaluate", evaluate_response)
graph.add_edge("refine", "evaluate")
graph.add_conditional_edges("evaluate", route_refinement, {"refine": "refine", END: END})
graph.set_entry_point("refine")

# Compile and run
app = graph.compile()
query = "Explain LangGraph in one sentence."
result = app.invoke({"messages": [{"role": "user", "content": query}], "iteration": 0})
print(f"Query: {query}\nFinal Answer: {result['messages'][-2]['content']}\n")

## 4. Memory Checkpoints

**Theory**: LangGraph's checkpointing feature allows saving and resuming graph states, enabling persistent workflows across sessions. This is useful for long-running tasks or resumable conversations. We'll create a checkpointed version of the customer support graph.

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint import MemorySaver
from langchain_community.llms import Ollama
from typing import TypedDict, Annotated
import operator

# Define state
class CheckpointState(TypedDict):
    messages: Annotated[list, operator.add]

# Initialize LLM
llm = Ollama(model="mistral")

# Define agent node
def support_agent(state: CheckpointState):
    """Handles customer support queries with conversation history."""
    messages = state["messages"]
    prompt = f"You are a customer support agent. Use the conversation history to answer:\n{messages}"
    response = llm.invoke(prompt)
    return {"messages": [{"content": response, "role": "assistant"}]}

# Create graph with checkpointing
graph = StateGraph(CheckpointState)
graph.add_node("agent", support_agent)
graph.add_edge("agent", END)
graph.set_entry_point("agent")

# Initialize memory saver
checkpointer = MemorySaver()

# Compile graph with checkpointing
app = graph.compile(checkpointer=checkpointer)

# Run conversation with checkpointing
thread_id = "support_thread_1"
queries = [
    "What is the status of Bob's order?",
    "Can he change the shipping address?"
]
for query in queries:
    result = app.invoke(
        {"messages": [{"content": query, "role": "user"}]},
        config={"configurable": {"thread_id": thread_id}}
    )
    print(f"Query: {query}\nAnswer: {result['messages'][-1]['content']}\n")

# Resume conversation later
resume_query = "What was the order status again?"
result = app.invoke(
    {"messages": [{"content": resume_query, "role": "user"}]},
    config={"configurable": {"thread_id": thread_id}}
)
print(f"Resume Query: {resume_query}\nAnswer: {result['messages'][-1]['content']}\n")

## 5. Human-in-the-Loop Interactions

**Theory**: LangGraph supports human-in-the-loop workflows by pausing execution at specific nodes to collect human input. This is useful for scenarios requiring human approval or clarification. We'll create a graph where a human reviews the agent's response before proceeding.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from typing import TypedDict, Annotated
import operator

# Define state
class HumanReviewState(TypedDict):
    messages: Annotated[list, operator.add]
    human_approved: bool

# Setup LLM
llm = ChatOllama(model="mixtral")

# Agent node
def agent(state: HumanReviewState):
    """Generates a response for human review."""
    user_msg = state["messages"][-1]["content"]
    response = llm.invoke(f"Answer as a support agent: {user_msg}")
    return {"messages": [{"role": "assistant", "content": response.content}]}

# Human review node (simulated)
def human_review(state: HumanReviewState):
    """Simulates human review by approving the response."""
    print(f"Human Review: Please review this response: {state['messages'][-1]['content']}")
    # Simulate human approval (in practice, this would be an input prompt)
    return {"human_approved": True}

# Conditional routing
def route_after_review(state: HumanReviewState):
    """Routes to END if human approves, else back to agent."""
    return END if state["human_approved"] else "agent"

# Create graph
graph = StateGraph(HumanReviewState)
graph.add_node("agent", agent)
graph.add_node("human", human_review)
graph.add_edge("agent", "human")
graph.add_conditional_edges("human", route_after_review, {"agent": "agent", END: END})
graph.set_entry_point("agent")

# Compile and run
app = graph.compile()
query = "Can I return my order?"
result = app.invoke({"messages": [{"role": "user", "content": query}], "human_approved": False})
print(f"Query: {query}\nFinal Answer: {result['messages'][-2]['content']}\n")

## 6. Custom State Management

**Theory**: LangGraph allows custom state schemas to track complex data beyond messages. This is useful for workflows requiring additional metadata, such as confidence scores or task status. We'll create a graph that tracks response confidence.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from typing import TypedDict, Annotated
import operator

# Define state
class ConfidenceState(TypedDict):
    messages: Annotated[list, operator.add]
    confidence: float

# Setup LLM
llm = ChatOllama(model="mixtral")

# Agent node
def confident_agent(state: ConfidenceState):
    """Generates a response and assigns a confidence score."""
    user_msg = state["messages"][-1]["content"]
    prompt = PromptTemplate.from_template(
        "Answer the query and provide a confidence score (0-1):\nQuery: {query}\nFormat: {{'answer': 'response', 'confidence': 0.x}}"
    )
    response = llm.invoke(prompt.format(query=user_msg))
    import json
    parsed = json.loads(response.content)
    return {
        "messages": [{"role": "assistant", "content": parsed["answer"]}],
        "confidence": parsed["confidence"]
    }

# Validator node
def validate_confidence(state: ConfidenceState):
    """Checks if the confidence score meets a threshold."""
    confidence = state["confidence"]
    return {"messages": [{"role": "validator", "content": f"Confidence: {confidence}"}] if confidence >= 0.8 else []}

# Conditional routing
def route_confidence(state: ConfidenceState):
    """Routes to END if confidence is high, else back to agent."""
    return END if state["confidence"] >= 0.8 else "agent"

# Create graph
graph = StateGraph(ConfidenceState)
graph.add_node("agent", confident_agent)
graph.add_node("validator", validate_confidence)
graph.add_edge("agent", "validator")
graph.add_conditional_edges("validator", route_confidence, {"agent": "agent", END: END})
graph.set_entry_point("agent")

# Compile and run
app = graph.compile()
query = "What is LangGraph?"
result = app.invoke({"messages": [{"role": "user", "content": query}], "confidence": 0.0})
print(f"Query: {query}\nAnswer: {result['messages'][-2]['content']}\nConfidence: {result['confidence']}\n")

## 7. Advanced Tool Integration

**Theory**: LangGraph supports advanced tool integration, allowing agents to interact with external APIs or services. We'll extend the calculator example with a web search tool for non-mathematical queries.

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_community.tools import DuckDuckGoSearchRun
from typing import TypedDict, Annotated
import operator

# Define state
class AdvancedAgentState(TypedDict):
    messages: Annotated[list, operator.add]

# Setup LLM and tools
llm = ChatOllama(model="mixtral")
search_tool = DuckDuckGoSearchRun()

# Calculator tool
def calculate(expression: str) -> str:
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

# Agent node
def advanced_agent(state: AdvancedAgentState):
    """Decides whether to use the calculator, search tool, or direct LLM response."""
    user_msg = state["messages"][-1]["content"]
    prompt = PromptTemplate.from_template(
        "Determine the action for this query:\nQuery: {query}\nOptions:\n1. Calculator: 'CALC: {query}'\n2. Search: 'SEARCH: {query}'\n3. Direct: 'DIRECT: {query}'"
    )
    response = llm.invoke(prompt.format(query=user_msg)).content
    return {"messages": [{"role": "assistant", "content": response}]}

# Tool node
def call_advanced_tool(state: AdvancedAgentState):
    """Executes the appropriate tool based on the agent's decision."""
    last_msg = state["messages"][-1]["content"]
    if last_msg.startswith("CALC:"):
        expression = last_msg.replace("CALC:", "").strip()
        result = calculate(expression)
        return {"messages": [{"role": "tool", "content": result}]}
    elif last_msg.startswith("SEARCH:"):
        query = last_msg.replace("SEARCH:", "").strip()
        result = search_tool.run(query)
        return {"messages": [{"role": "tool", "content": result}]}
    elif last_msg.startswith("DIRECT:"):
        query = last_msg.replace("DIRECT:", "").strip()
        result = llm.invoke(query).content
        return {"messages": [{"role": "tool", "content": result}]}
    return {"messages": []}

# Create graph
graph = StateGraph(AdvancedAgentState)
graph.add_node("agent", advanced_agent)
graph.add_node("tool", call_advanced_tool)
graph.add_edge("agent", "tool")
graph.add_conditional_edges("tool", lambda state: "agent" if state["messages"] else END)
graph.set_entry_point("agent")

# Compile and run
app = graph.compile()
queries = ["What is 10 / 2?", "What is LangGraph?", "Latest AI news"]
for query in queries:
    result = app.invoke({"messages": [{"role": "user", "content": query}]})
    print(f"Query: {query}\nAnswer: {result['messages'][-1]['content']}\n")

## Conclusion

This notebook covers the core and advanced functionalities of LangGraph, including basic graphs, cyclical workflows, memory checkpoints, human-in-the-loop interactions, custom state management, and advanced tool integration. It preserves the original calculator and customer support examples while adding new examples to demonstrate LangGraph's flexibility.

For further exploration, refer to the [LangGraph Documentation](https://langchain-ai.github.io/langgraph/) and experiment with different LLMs, tools, and state schemas.